In [ ]:
import os
import sys
sys.path.append("../")
sys.path.append("../..")
sys.path.append("../intervention-estimation/")
sys.path.append("../../src")
import os
import pickle

from functions import run_ours_real

import numpy as np
from causaldag import unknown_target_igsp
from causaldag import partial_correlation_test, MemoizedCI_Tester, partial_correlation_suffstat
from causaldag import MemoizedInvarianceTester, gauss_invariance_test, gauss_invariance_suffstat
from src.tools.metric import get_compared_components, get_skeleton, metric_skeleton_level, metric_cpdag_level

In [ ]:
def metric_target_level_for_utigsp(pred, targ):
    """ P / R / F1 (I-TARGET Level) """
    TP , TP_FP, TP_FN = 0, 0, 0
    for set_pred, set_targ in zip(pred, targ):
        TP += len(set_pred.intersection(set_targ))
        TP_FP += len(set_pred)
        TP_FN += len(set_targ)
    precision = TP / max(TP_FP, 1)
    recall = TP / max(TP_FN, 1)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall != 0 else 0
    pred_iden_edges_num, targ_iden_edges_num = TP_FP, TP_FN
    return {'p':round(precision,2), 'r':round(recall,2), 'f1':round(f1,2), '#pred_iden_edges':int(pred_iden_edges_num), '#targ_iden_edges':int(targ_iden_edges_num)}

In [ ]:
exp_name = 'exp_200'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../exps_of_result/ut-igsp/{exp_name}/know', exist_ok=True)
os.makedirs(f'../exps_of_result/ut-igsp/{exp_name}/unknow', exist_ok=True)


for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    try:
        know_pred_graph_path = f'../exps_of_result/ut-igsp/{exp_name}/know/{benchmark_name}_aug_graph.txt'
        unknow_pred_graph_path = f'../exps_of_result/ut-igsp/{exp_name}/unknow/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        obs_samples, iv_samples_list = data_list[0], data_list[1:]
        nodes = set(range(obs_samples.shape[1]))
        obs_suffstat = partial_correlation_suffstat(obs_samples)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=1e-3)
        invariance_suffstat = gauss_invariance_suffstat(obs_samples, iv_samples_list)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=1e-3)
        
        unknow_setting_list = [dict(known_interventions=[]) for _ in know_targets_list[1:]]
        know_setting_list = [dict(known_interventions=set(targets), samples=iv_samples_list[idx]) for idx, targets in enumerate(know_targets_list[1:])]
        
        S_obs = (obs_samples.T@obs_samples)/obs_samples.shape[0]
        S_int = {}
        for idx_setting in range(len(know_setting_list)):
            samples_current = know_setting_list[idx_setting]['samples']
            S_current = (samples_current.T @ samples_current)/samples_current.shape[0]
            S_int['setting_%d'%idx_setting] = S_current
        
        lambda_l1 = 0.1
        # for J0 threshold
        single_threshold = 0.05
        # for building J0 descendants
        pair_l1 = 0.05
        # remove the small values after J0 descendants built
        pair_threshold = 0.005
        # always taken one. ADMM parameter
        rho = 1.0

        # this is more important one. Penalty parameter for parent selection
        parent_l1_list = [0.005,0.01,0.02,0.03,0.04,0.05,0.06,0.08,0.09,0.10]
        
        save_min = 0
        save_mt = {}
        for parent_l1 in parent_l1_list:
            est_cpdag, est_skeleton, I_hat_all, I_hat_parents_all, Ij_hat_parents_all, N_lists_all, A_groups_all, time_all = \
            run_ours_real(S_obs,S_int,lambda_l1,single_threshold,pair_l1,pair_threshold, parent_l1,rho)
            pred_I_TARGETS_unknow = [set(value) for key, value in I_hat_all.items()]
        
            mt_target_unknow = metric_target_level_for_utigsp(pred_I_TARGETS_unknow, list(map(set, know_targets_list[1:])))
            if mt_target_unknow['f1'] > save_min:
                save_min = mt_target_unknow['f1']
                save_mt = mt_target_unknow
        print(f'unknow performance: \n targets:{save_mt} \n')

    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_1_1'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../exps_of_result/ut-igsp/{exp_name}/know', exist_ok=True)
os.makedirs(f'../baselines/exps_of_result/ut-igsp/{exp_name}/unknow', exist_ok=True)


for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    try:
        know_pred_graph_path = f'../exps_of_result/ut-igsp/{exp_name}/know/{benchmark_name}_aug_graph.txt'
        unknow_pred_graph_path = f'../exps_of_result/ut-igsp/{exp_name}/unknow/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        obs_samples, iv_samples_list = data_list[0], data_list[1:]
        nodes = set(range(obs_samples.shape[1]))
        obs_suffstat = partial_correlation_suffstat(obs_samples)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=1e-3)
        invariance_suffstat = gauss_invariance_suffstat(obs_samples, iv_samples_list)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=1e-3)
        
        unknow_setting_list = [dict(known_interventions=[]) for _ in know_targets_list[1:]]
        know_setting_list = [dict(known_interventions=set(targets), samples=iv_samples_list[idx]) for idx, targets in enumerate(know_targets_list[1:])]
        
        S_obs = (obs_samples.T@obs_samples)/obs_samples.shape[0]
        S_int = {}
        for idx_setting in range(len(know_setting_list)):
            samples_current = know_setting_list[idx_setting]['samples']
            S_current = (samples_current.T @ samples_current)/samples_current.shape[0]
            S_int['setting_%d'%idx_setting] = S_current
        
        lambda_l1 = 0.1
        # for J0 threshold
        single_threshold = 0.05
        # for building J0 descendants
        pair_l1 = 0.05
        # remove the small values after J0 descendants built
        pair_threshold = 0.005
        # always taken one. ADMM parameter
        rho = 1.0

        # this is more important one. Penalty parameter for parent selection
        parent_l1_list = [0.005,0.01,0.02,0.03,0.04,0.05,0.06,0.08,0.09,0.10]
        
        save_min = 0
        save_mt = {}
        for parent_l1 in parent_l1_list:
            est_cpdag, est_skeleton, I_hat_all, I_hat_parents_all, Ij_hat_parents_all, N_lists_all, A_groups_all, time_all = \
            run_ours_real(S_obs,S_int,lambda_l1,single_threshold,pair_l1,pair_threshold, parent_l1,rho)
            pred_I_TARGETS_unknow = [set(value) for key, value in I_hat_all.items()]
        
            mt_target_unknow = metric_target_level_for_utigsp(pred_I_TARGETS_unknow, list(map(set, know_targets_list[1:])))
            if mt_target_unknow['f1'] > save_min:
                save_min = mt_target_unknow['f1']
                save_mt = mt_target_unknow
        print(f'unknow performance: \n targets:{save_mt} \n')

    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_1_1'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../exps_of_result/ut-igsp/{exp_name}/know', exist_ok=True)
os.makedirs(f'../baselines/exps_of_result/ut-igsp/{exp_name}/unknow', exist_ok=True)


for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx!=0:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    try:
        know_pred_graph_path = f'../baselines/exps_of_result/ut-igsp/{exp_name}/know/{benchmark_name}_aug_graph.txt'
        unknow_pred_graph_path = f'../baselines/exps_of_result/ut-igsp/{exp_name}/unknow/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        obs_samples, iv_samples_list = data_list[0], data_list[1:]
        nodes = set(range(obs_samples.shape[1]))
        obs_suffstat = partial_correlation_suffstat(obs_samples)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=1e-3)
        invariance_suffstat = gauss_invariance_suffstat(obs_samples, iv_samples_list)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=1e-3)
        
        unknow_setting_list = [dict(known_interventions=[]) for _ in know_targets_list[1:]]
        know_setting_list = [dict(known_interventions=set(targets), samples=iv_samples_list[idx]) for idx, targets in enumerate(know_targets_list[1:])]
        
        S_obs = (obs_samples.T@obs_samples)/obs_samples.shape[0]
        S_int = {}
        for idx_setting in range(len(know_setting_list)):
            samples_current = know_setting_list[idx_setting]['samples']
            S_current = (samples_current.T @ samples_current)/samples_current.shape[0]
            S_int['setting_%d'%idx_setting] = S_current
        
        lambda_l1 = 0.1
        # for J0 threshold
        single_threshold = 0.05
        # for building J0 descendants
        pair_l1 = 0.05
        # remove the small values after J0 descendants built
        pair_threshold = 0.005
        # always taken one. ADMM parameter
        rho = 1.0

        # this is more important one. Penalty parameter for parent selection
        parent_l1_list = [0.005,0.01,0.02,0.03,0.04,0.05,0.06,0.08,0.09,0.10]
        
        save_min = 0
        save_mt = {}
        for parent_l1 in parent_l1_list:
            est_cpdag, est_skeleton, I_hat_all, I_hat_parents_all, Ij_hat_parents_all, N_lists_all, A_groups_all, time_all = \
            run_ours_real(S_obs,S_int,lambda_l1,single_threshold,pair_l1,pair_threshold, parent_l1,rho)
            pred_I_TARGETS_unknow = [set(value) for key, value in I_hat_all.items()]
        
            mt_target_unknow = metric_target_level_for_utigsp(pred_I_TARGETS_unknow, list(map(set, know_targets_list[1:])))
            if mt_target_unknow['f1'] > save_min:
                save_min = mt_target_unknow['f1']
                save_mt = mt_target_unknow
        print(f'unknow performance: \n targets:{save_mt} \n')

    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_1_2'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../exps_of_result/crte/{exp_name}/know', exist_ok=True)
os.makedirs(f'../baselines/exps_of_result/crte/{exp_name}/unknow', exist_ok=True)


for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx>=7:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    try:
        know_pred_graph_path = f'../baselines/exps_of_result/crte/{exp_name}/know/{benchmark_name}_aug_graph.txt'
        unknow_pred_graph_path = f'../baselines/exps_of_result/crte/{exp_name}/unknow/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        obs_samples, iv_samples_list = data_list[0], data_list[1:]
        nodes = set(range(obs_samples.shape[1]))
        obs_suffstat = partial_correlation_suffstat(obs_samples)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=1e-3)
        invariance_suffstat = gauss_invariance_suffstat(obs_samples, iv_samples_list)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=1e-3)
        
        unknow_setting_list = [dict(known_interventions=[]) for _ in know_targets_list[1:]]
        know_setting_list = [dict(known_interventions=set(targets), samples=iv_samples_list[idx]) for idx, targets in enumerate(know_targets_list[1:])]
        
        S_obs = (obs_samples.T@obs_samples)/obs_samples.shape[0]
        S_int = {}
        for idx_setting in range(len(know_setting_list)):
            samples_current = know_setting_list[idx_setting]['samples']
            S_current = (samples_current.T @ samples_current)/samples_current.shape[0]
            S_int['setting_%d'%idx_setting] = S_current
        
        lambda_l1 = 0.1
        # for J0 threshold
        single_threshold = 0.05
        # for building J0 descendants
        pair_l1 = 0.05
        # remove the small values after J0 descendants built
        pair_threshold = 0.005
        # always taken one. ADMM parameter
        rho = 1.0

        # this is more important one. Penalty parameter for parent selection
        parent_l1_list = [0.005,0.01,0.02,0.03,0.04,0.05,0.06,0.08,0.09,0.10]
        
        save_min = 0
        save_mt = {}
        for parent_l1 in parent_l1_list:
            est_cpdag, est_skeleton, I_hat_all, I_hat_parents_all, Ij_hat_parents_all, N_lists_all, A_groups_all, time_all = \
            run_ours_real(S_obs,S_int,lambda_l1,single_threshold,pair_l1,pair_threshold, parent_l1,rho)
            pred_I_TARGETS_unknow = [set(value) for key, value in I_hat_all.items()]
        
            mt_target_unknow = metric_target_level_for_utigsp(pred_I_TARGETS_unknow, list(map(set, know_targets_list[1:])))
            if mt_target_unknow['f1'] > save_min:
                save_min = mt_target_unknow['f1']
                save_mt = mt_target_unknow
        print(f'unknow performance: \n targets:{save_mt} \n')

    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_2_5'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../exps_of_result/crte/{exp_name}/know', exist_ok=True)
os.makedirs(f'../baselines/exps_of_result/crte/{exp_name}/unknow', exist_ok=True)


for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx>=7:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    try:
        know_pred_graph_path = f'../baselines/exps_of_result/crte/{exp_name}/know/{benchmark_name}_aug_graph.txt'
        unknow_pred_graph_path = f'../baselines/exps_of_result/crte/{exp_name}/unknow/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        obs_samples, iv_samples_list = data_list[0], data_list[1:]
        nodes = set(range(obs_samples.shape[1]))
        obs_suffstat = partial_correlation_suffstat(obs_samples)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=1e-3)
        invariance_suffstat = gauss_invariance_suffstat(obs_samples, iv_samples_list)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=1e-3)
        
        unknow_setting_list = [dict(known_interventions=[]) for _ in know_targets_list[1:]]
        know_setting_list = [dict(known_interventions=set(targets), samples=iv_samples_list[idx]) for idx, targets in enumerate(know_targets_list[1:])]
        
        S_obs = (obs_samples.T@obs_samples)/obs_samples.shape[0]
        S_int = {}
        for idx_setting in range(len(know_setting_list)):
            samples_current = know_setting_list[idx_setting]['samples']
            S_current = (samples_current.T @ samples_current)/samples_current.shape[0]
            S_int['setting_%d'%idx_setting] = S_current
        
        lambda_l1 = 0.1
        # for J0 threshold
        single_threshold = 0.05
        # for building J0 descendants
        pair_l1 = 0.05
        # remove the small values after J0 descendants built
        pair_threshold = 0.005
        # always taken one. ADMM parameter
        rho = 1.0

        # this is more important one. Penalty parameter for parent selection
        parent_l1_list = [0.005,0.01,0.02,0.03,0.04,0.05,0.06,0.08,0.09,0.10]
        
        save_min = 0
        save_mt = {}
        for parent_l1 in parent_l1_list:
            est_cpdag, est_skeleton, I_hat_all, I_hat_parents_all, Ij_hat_parents_all, N_lists_all, A_groups_all, time_all = \
            run_ours_real(S_obs,S_int,lambda_l1,single_threshold,pair_l1,pair_threshold, parent_l1,rho)
            pred_I_TARGETS_unknow = [set(value) for key, value in I_hat_all.items()]
        
            mt_target_unknow = metric_target_level_for_utigsp(pred_I_TARGETS_unknow, list(map(set, know_targets_list[1:])))
            if mt_target_unknow['f1'] > save_min:
                save_min = mt_target_unknow['f1']
                save_mt = mt_target_unknow
        print(f'unknow performance: \n targets:{save_mt} \n')

    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_2_8'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../exps_of_result/crte/{exp_name}/know', exist_ok=True)
os.makedirs(f'../baselines/exps_of_result/crte/{exp_name}/unknow', exist_ok=True)


for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx>=7:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    try:
        know_pred_graph_path = f'../exps_of_result/crte/{exp_name}/know/{benchmark_name}_aug_graph.txt'
        unknow_pred_graph_path = f'../exps_of_result/crte/{exp_name}/unknow/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        obs_samples, iv_samples_list = data_list[0], data_list[1:]
        nodes = set(range(obs_samples.shape[1]))
        obs_suffstat = partial_correlation_suffstat(obs_samples)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=1e-3)
        invariance_suffstat = gauss_invariance_suffstat(obs_samples, iv_samples_list)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=1e-3)
        
        unknow_setting_list = [dict(known_interventions=[]) for _ in know_targets_list[1:]]
        know_setting_list = [dict(known_interventions=set(targets), samples=iv_samples_list[idx]) for idx, targets in enumerate(know_targets_list[1:])]
        
        S_obs = (obs_samples.T@obs_samples)/obs_samples.shape[0]
        S_int = {}
        for idx_setting in range(len(know_setting_list)):
            samples_current = know_setting_list[idx_setting]['samples']
            S_current = (samples_current.T @ samples_current)/samples_current.shape[0]
            S_int['setting_%d'%idx_setting] = S_current
        
        lambda_l1 = 0.1
        # for J0 threshold
        single_threshold = 0.05
        # for building J0 descendants
        pair_l1 = 0.05
        # remove the small values after J0 descendants built
        pair_threshold = 0.005
        # always taken one. ADMM parameter
        rho = 1.0

        # this is more important one. Penalty parameter for parent selection
        parent_l1_list = [0.005,0.01,0.02,0.03,0.04,0.05,0.06,0.08,0.09,0.10]
        
        save_min = 0
        save_mt = {}
        for parent_l1 in parent_l1_list:
            est_cpdag, est_skeleton, I_hat_all, I_hat_parents_all, Ij_hat_parents_all, N_lists_all, A_groups_all, time_all = \
            run_ours_real(S_obs,S_int,lambda_l1,single_threshold,pair_l1,pair_threshold, parent_l1,rho)
            pred_I_TARGETS_unknow = [set(value) for key, value in I_hat_all.items()]
        
            mt_target_unknow = metric_target_level_for_utigsp(pred_I_TARGETS_unknow, list(map(set, know_targets_list[1:])))
            if mt_target_unknow['f1'] > save_min:
                save_min = mt_target_unknow['f1']
                save_mt = mt_target_unknow
        print(f'unknow performance: \n targets:{save_mt} \n')

    except:
        print(f'pass {benchmark_name}\n')